In [0]:
from pyspark.sql import functions as F

In [0]:
# List files in the volume to understand what we're working with
volume_path = "/Volumes/turbine_poc/wind_farm/landing/source_data"

import os
files = dbutils.fs.ls(volume_path)
for file in files:
    print(f"{file.name} - Size: {file.size} bytes")

In [0]:
# Load all 3 CSV files from the volume into a single dataframe
df = spark.read.csv(
    f"{volume_path}/*.csv",
    header=True,
    inferSchema=True
)

# Display basic info about the combined dataframe
print(f"Total rows: {df.count():,}")
print(f"Total columns: {len(df.columns)}")
print("\nSchema:")
df.printSchema()
print("\nFirst few rows:")
display(df.limit(10))

In [0]:
#duplicates
print(df.count())
print(df.dropDuplicates().count())

In [0]:
display(df.describe())

In [0]:
cast_check = df.select(
    F.sum(F.when(F.col("timestamp").isNotNull() &
                 F.to_timestamp("timestamp").isNull(), 1).otherwise(0)).alias("bad_timestamp"),
    F.sum(F.when(F.col("turbine_id").isNotNull() &
                 F.col("turbine_id").cast("int").isNull(), 1).otherwise(0)).alias("bad_turbine_id"),
    F.sum(F.when(F.col("wind_speed").isNotNull() &
                 F.col("wind_speed").cast("double").isNull(), 1).otherwise(0)).alias("bad_wind_speed"),
    F.sum(F.when(F.col("wind_direction").isNotNull() &
                 F.col("wind_direction").cast("int").isNull(), 1).otherwise(0)).alias("bad_wind_direction"),
    F.sum(F.when(F.col("power_output").isNotNull() &
                 F.col("power_output").cast("double").isNull(), 1).otherwise(0)).alias("bad_power_output"),
)
display(cast_check)


In [0]:
# Correlation analysis between wind speed and power output
from pyspark.sql.functions import corr

# Calculate correlation
correlation = df.select(corr("wind_speed", "power_output")).collect()[0][0]
print(f"Correlation between wind_speed and power_output: {correlation:.4f}")

# Show correlation matrix for all numeric columns
print("\nFull correlation analysis:")
for col1 in ["wind_speed", "wind_direction", "power_output"]:
    for col2 in ["wind_speed", "wind_direction", "power_output"]:
        if col1 != col2:
            corr_val = df.select(corr(col1, col2)).collect()[0][0]
            print(f"{col1} vs {col2}: {corr_val:.4f}")

In [0]:
# Create wind-band table to visualize power curve
# Group wind speeds into 1 m/s bands

wind_band_df = df.withColumn(
    "wind_band", 
    F.floor(F.col("wind_speed")).cast("int")
).groupBy("wind_band").agg(
    F.avg("power_output").alias("avg_power_output"),
    F.min("power_output").alias("min_power_output"),
    F.max("power_output").alias("max_power_output"),
    F.stddev("power_output").alias("stddev_power_output"),
    F.count("*").alias("observation_count")
).orderBy("wind_band")

print("Wind-Band Power Curve Table:")
display(wind_band_df)

In [0]:
# Visualize the power curve
import matplotlib.pyplot as plt

# Convert to pandas for plotting
wind_band_pd = wind_band_df.toPandas()

plt.figure(figsize=(12, 6))

# Plot average power output with error bars (showing min/max range)
plt.subplot(1, 2, 1)
plt.plot(wind_band_pd['wind_band'], wind_band_pd['avg_power_output'], 'o-', linewidth=2, markersize=8)
plt.fill_between(wind_band_pd['wind_band'], 
                  wind_band_pd['min_power_output'], 
                  wind_band_pd['max_power_output'], 
                  alpha=0.3)
plt.xlabel('Wind Speed (m/s)', fontsize=12)
plt.ylabel('Power Output (MW)', fontsize=12)
plt.title('Wind Turbine Power Curve', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Plot observation counts per wind band
plt.subplot(1, 2, 2)
plt.bar(wind_band_pd['wind_band'], wind_band_pd['observation_count'], color='steelblue')
plt.xlabel('Wind Speed (m/s)', fontsize=12)
plt.ylabel('Number of Observations', fontsize=12)
plt.title('Data Distribution by Wind Speed', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [0]:
# Do all the turbines see the same weather at the same time?
#
# If they are on one site they should. At any given hour they would all be in
# similar wind, so most of the variation in the data would come from the hour
# changing, not from the turbines differing from each other.
#
# I split the variance into those two parts. No threshold needed here: a ratio
# of 1.0 means the hour explains nothing.

def explained_by(data, group_col, value_col):
    total  = data.select(F.var_samp(value_col)).first()[0]
    within = (data.groupBy(group_col)
                  .agg(F.var_samp(value_col).alias("v"))
                  .select(F.avg("v")).first()[0])
    return 1 - within / total


for col in ["wind_speed", "power_output"]:
    print(f"{col:14s} explained by the hour: {explained_by(df, 'timestamp', col):.2%}")

# A real wind farm would be high here. Near zero means each turbine has its own
# independent wind, which is not how a wind farm works.


In [0]:
# Sanity check on the result above.
#
# I deal every reading out to a random hour, which destroys any link between a
# reading and when it was taken, then run exactly the same test. If the real
# timestamps score the same as invented ones, they were carrying no information
# in the first place.

fake = df.withColumn("fake_hour", (F.rand(79) * 744).cast("int"))

print(f"{'measure':<16}{'real hours':>12}{'random hours':>14}")
print("-" * 42)
for col in ["wind_speed", "power_output"]:
    real = explained_by(df, "timestamp", col)
    rand = explained_by(fake, "fake_hour", col)
    print(f"{col:<16}{real:>12.2%}{rand:>14.2%}")


In [0]:
# And one more way of looking at it: correlate every pair of turbines.
#
# Clusters of turbines behaving alike would show up as pairs that rise and fall
# together. The noise level tells me what counts as zero for this sample size,
# so I do not have to pick a cutoff myself.

import numpy as np

pivot = df.groupBy("timestamp").pivot("turbine_id").agg(F.first("wind_speed")).na.drop()
corr  = pivot.drop("timestamp").toPandas().corr().values
pairs = corr[np.triu_indices_from(corr, k=1)]

n_hours = pivot.count()
noise   = 1.96 / np.sqrt(n_hours)

print(f"pairs compared : {len(pairs)}")
print(f"strongest pair : {np.abs(pairs).max():.4f}")
print(f"noise level    : {noise:.4f}  (95% band for a true zero at n={n_hours})")
print(f"above the band : {(np.abs(pairs) > noise).sum()} of {len(pairs)}"
      f"   (expect about {int(0.05*len(pairs))} by chance)")


In [0]:
# Distribution Analysis - Check if data is synthetically generated
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Convert to pandas for plotting
df_pd = df.select("wind_speed", "wind_direction", "power_output").toPandas()

# Wind Speed Distribution
axes[0, 0].hist(df_pd['wind_speed'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Wind Speed (m/s)', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Wind Speed Distribution', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Power Output Distribution
axes[0, 1].hist(df_pd['power_output'], bins=50, color='green', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Power Output (MW)', fontsize=11)
axes[0, 1].set_ylabel('Frequency', fontsize=11)
axes[0, 1].set_title('Power Output Distribution', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Wind Direction Distribution
axes[1, 0].hist(df_pd['wind_direction'], bins=36, color='orange', edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Wind Direction (degrees)', fontsize=11)
axes[1, 0].set_ylabel('Frequency', fontsize=11)
axes[1, 0].set_title('Wind Direction Distribution', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Wind Direction Polar Plot
ax_polar = plt.subplot(2, 2, 4, projection='polar')
bins = np.linspace(0, 2*np.pi, 37)
theta = np.radians(df_pd['wind_direction'])
ax_polar.hist(theta, bins=bins, color='coral', edgecolor='black', alpha=0.7)
ax_polar.set_theta_zero_location('N')
ax_polar.set_theta_direction(-1)
ax_polar.set_title('Wind Direction (Polar)', fontsize=12, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

# The histograms are flat. What matters is what that costs me later, so I measure it.
#
# For a uniform distribution the whole range is sqrt(12) = 3.46 standard deviations
# wide, so the furthest any single reading can sit from the mean is half of that:
# 1.73 sigma. The brief asks me to flag readings more than 2 sigma out. On this data
# that rule can never fire - not because nothing is wrong, but because the shape of
# the distribution forbids it.

# The histograms are flat. What matters is what that costs me later, so I measure it.
#
# For a uniform distribution the whole range is sqrt(12) = 3.46 standard deviations
# wide, so the furthest any single reading can sit from the mean is half of that:
# 1.73 sigma. The brief asks me to flag readings more than 2 sigma out. On this data
# that rule can never fire - not because nothing is wrong, but because the shape of
# the distribution forbids it.

print("\n=== Distribution Analysis ===")

for col, unit in [("wind_speed", "m/s"), ("power_output", "MW"), ("wind_direction", "deg")]:
    s = df_pd[col]
    span_in_sigmas = (s.max() - s.min()) / s.std()
    worst_z = max(abs(s.max() - s.mean()), abs(s.mean() - s.min())) / s.std()
    print(f"\n{col}")
    print(f"  mean {s.mean():.2f} {unit}, std {s.std():.2f}, range [{s.min():.2f}, {s.max():.2f}]")
    print(f"  range spans {span_in_sigmas:.2f} sigmas   (3.46 = textbook uniform)")
    print(f"  furthest possible reading is {worst_z:.2f} sigma from the mean")

print("\nConclusion")
print("Flat histograms, every column spans ~3.46 sigmas - textbook uniform.")
print("No single reading can reach 2 sigma, so the rule cannot be applied to readings.")
print("On daily averages it does fire: 17 of 465 turbine-days, 3.7%, which is just the")
print("tail a 2-sigma threshold cuts off any population. Nothing is broken in this data,")
print("so there is nothing for the rule to find - it returns the extremes of noise.")
print("That is why I wrote a generator: inject a fault I can point at, then check the")
print("pipeline finds it. Turbine 7 lands at z = -3.04, clear of the -2.63 worst noise.")

In [0]:
# CONCLUSION
#
# The turbines do not share weather in this data. The hour explains none of the
# variation, randomly shuffled timestamps score exactly the same, and no pair of
# turbines moves together beyond what chance gives me.
#
# So I cannot build a rule that says "these turbines should agree with each other
# right now" - it would fire on almost every row.
#
# What I can still do is compare a turbine against the fleet, because all 15 are
# drawing from the same distribution. That makes it a statistical test rather than
# a physical one. On real telemetry the physical argument would hold as well, but
# here I should say what it actually is.
